In [42]:
# merging the data
import os
from cv2 import merge
import pandas as pd
import json
    
raw_sensor_data_dir = r"E:\src\neat-calculator\data_collection\recordings\sensor_data\SensorRecording\08022026"
sensor_data_merged_path = r"E:\src\neat-calculator\ml_monolith\data_preprocessing\merged_sensor_data_08022026.json"
raw_labels_dir = r"E:\src\neat-calculator\ml_monolith\data_preprocessing\label_files_08022026"
labels_merged_path = r"E:\src\neat-calculator\ml_monolith\data_preprocessing\merged_labels_08022026.csv"

def merge_json_files(json_dir):

    merged_data = []
    files_sorted = sorted(os.listdir(json_dir))
    for filename in files_sorted:
        if filename.endswith('.json'):
            with open(os.path.join(json_dir, filename), 'r') as f:
                data = json.load(f)
                merged_data.extend(data)
    return merged_data

def merge_csv_files(csv_dir):
    merged_df = pd.DataFrame()
    files_sorted = sorted(os.listdir(csv_dir))
    for filename in files_sorted:
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(csv_dir, filename))
            merged_df = pd.concat([merged_df, df], ignore_index=True)
    return merged_df



merged_data = merge_json_files(raw_sensor_data_dir)
print(f"Saving merged data to: {sensor_data_merged_path}")
with open(sensor_data_merged_path, 'w') as f:
    json.dump(merged_data, f)   
    
print(f"Saving merged labels to: {labels_merged_path}")
merged_labels_df = merge_csv_files(raw_labels_dir)
merged_labels_df.to_csv(labels_merged_path, index=False)



# load labels from csv file

Saving merged data to: E:\src\neat-calculator\ml_monolith\data_preprocessing\merged_sensor_data_08022026.json
Saving merged labels to: E:\src\neat-calculator\ml_monolith\data_preprocessing\merged_labels_08022026.csv


In [43]:
print(sensor_data_merged_path)
print(labels_merged_path)

# Read JSON without automatic date parsing to preserve Unix timestamps
sensor_data_df = pd.read_json(sensor_data_merged_path, convert_dates=False)
labels_df = pd.read_csv(labels_merged_path)

print(sensor_data_df.head())
print(f"Data types:\n{sensor_data_df.dtypes}")
print(f"\nFirst few timestamp values (Unix epoch ms): {sensor_data_df['timestamp'].head().tolist()}")


E:\src\neat-calculator\ml_monolith\data_preprocessing\merged_sensor_data_08022026.json
E:\src\neat-calculator\ml_monolith\data_preprocessing\merged_labels_08022026.csv
   accelerometerX  accelerometerY  accelerometerZ  gyroscopeX  gyroscopeY  \
0           16.82      -16.490000           -4.50        1.45       -1.09   
1           -6.53       -6.300000            2.60        0.23       -2.03   
2           -6.53       -5.710000            5.18       -0.33       -4.63   
3           -6.31       -6.340000            4.25       -0.60       -4.76   
4           -3.70      -11.809999            1.95        0.60       -1.18   

   gyroscopeZ      timestamp    timestampNanos  
0        0.08  1770556241638  1104971909198214  
1       -2.34  1770556241658  1104971929044829  
2       -2.03  1770556241678  1104971949366964  
3       -1.24  1770556241699  1104971958936287  
4        1.23  1770556241737  1104972009120037  
Data types:
accelerometerX    float64
accelerometerY    float64
acceleromet

In [44]:
print(labels_df.head())

         Label  StartTimestamp_Unix_Ms  EndTimestamp_Unix_Ms
0  Stairs Down           1770556164324         1770556199074
1      Walking           1770556208426         1770556242166
2      Walking           1770557152264         1770557224330
3    Stairs Up           1770557436738         1770557482746
4     Standing           1770557597454         1770557632976


In [45]:
# Quick validation before running the labeling
print("=== PRE-LABELING VALIDATION ===")

print(f"Labels to process:")
for i, row in labels_df.iterrows():
    duration_ms = row['EndTimestamp_Unix_Ms'] - row['StartTimestamp_Unix_Ms'] 
    duration_sec = duration_ms / 1000
    print(f"  {row['Label']}: {duration_sec:.1f}s ({row['StartTimestamp_Unix_Ms']} -> {row['EndTimestamp_Unix_Ms']})")

print(f"\nSensor data timestamp range:")
print(f"  First: {sensor_data_df['timestamp'].min()}")
print(f"  Last:  {sensor_data_df['timestamp'].max()}")

# Check for timestamp overlaps
print(f"\nChecking timestamp alignment...")
for i, row in labels_df.iterrows():
    sensor_count = ((sensor_data_df['timestamp'] >= row['StartTimestamp_Unix_Ms']) & 
                   (sensor_data_df['timestamp'] <= row['EndTimestamp_Unix_Ms'])).sum()
    print(f"  {row['Label']}: {sensor_count} sensor samples overlap")

print("\n" + "="*50)

=== PRE-LABELING VALIDATION ===
Labels to process:
  Stairs Down: 34.8s (1770556164324 -> 1770556199074)
  Walking: 33.7s (1770556208426 -> 1770556242166)
  Walking: 72.1s (1770557152264 -> 1770557224330)
  Stairs Up: 46.0s (1770557436738 -> 1770557482746)
  Standing: 35.5s (1770557597454 -> 1770557632976)
  Sitting: 50.7s (1770557635658 -> 1770557686372)
  Laying: 41.6s (1770557695440 -> 1770557737050)
  Stairs Down: 36.8s (1770561115962 -> 1770561152789)
  Stairs Up: 5.0s (1770561154802 -> 1770561159787)
  Stairs Up: 35.0s (1770561166589 -> 1770561201630)

Sensor data timestamp range:
  First: 1770556157833
  Last:  1770561221448

Checking timestamp alignment...
  Stairs Down: 445 sensor samples overlap
  Walking: 1259 sensor samples overlap
  Walking: 371 sensor samples overlap
  Stairs Up: 27 sensor samples overlap
  Standing: 275 sensor samples overlap
  Sitting: 340 sensor samples overlap
  Laying: 381 sensor samples overlap
  Stairs Down: 457 sensor samples overlap
  Stairs Up: 

In [46]:
# Initialize Activity column with None
sensor_data_df['Activity'] = None

print(f"Total sensor data points: {len(sensor_data_df)}")
print(f"Timestamp range: {sensor_data_df['timestamp'].min()} to {sensor_data_df['timestamp'].max()}")
print(f"Processing {len(labels_df)} labels...")

for index, row in labels_df.iterrows():
    # Calculate buffered timestamps (5 seconds = 5000ms)
    start_timestamp = row['StartTimestamp_Unix_Ms'] + 5000
    end_timestamp = row['EndTimestamp_Unix_Ms'] - 5000
    
    print(f"\nLabel {index+1}: {row['Label']}")
    print(f"  Original range: {row['StartTimestamp_Unix_Ms']} to {row['EndTimestamp_Unix_Ms']}")
    print(f"  Buffered range: {start_timestamp} to {end_timestamp}")
    
    # Use boolean indexing to update only matching rows
    mask = (sensor_data_df['timestamp'] >= start_timestamp) & (sensor_data_df['timestamp'] <= end_timestamp)
    matching_rows = mask.sum()
    
    if matching_rows > 0:
        sensor_data_df.loc[mask, 'Activity'] = row['Label']
        print(f"  ✓ Labeled {matching_rows} sensor data points")
    else:
        print(f"  ⚠ No matching sensor data found!")

print(f"\nLabeling complete!")
sensor_data_df.info()

Total sensor data points: 9523
Timestamp range: 1770556157833 to 1770561221448
Processing 10 labels...

Label 1: Stairs Down
  Original range: 1770556164324 to 1770556199074
  Buffered range: 1770556169324 to 1770556194074
  ✓ Labeled 226 sensor data points

Label 2: Walking
  Original range: 1770556208426 to 1770556242166
  Buffered range: 1770556213426 to 1770556237166
  ✓ Labeled 894 sensor data points

Label 3: Walking
  Original range: 1770557152264 to 1770557224330
  Buffered range: 1770557157264 to 1770557219330
  ✓ Labeled 166 sensor data points

Label 4: Stairs Up
  Original range: 1770557436738 to 1770557482746
  Buffered range: 1770557441738 to 1770557477746
  ✓ Labeled 27 sensor data points

Label 5: Standing
  Original range: 1770557597454 to 1770557632976
  Buffered range: 1770557602454 to 1770557627976
  ✓ Labeled 120 sensor data points

Label 6: Sitting
  Original range: 1770557635658 to 1770557686372
  Buffered range: 1770557640658 to 1770557681372
  ✓ Labeled 237 sens

In [47]:
# Verify labeling results
print("Activity distribution:")
activity_counts = sensor_data_df['Activity'].value_counts(dropna=False)
print(activity_counts)

print(f"\nLabeling coverage:")
labeled_count = sensor_data_df['Activity'].notna().sum()
total_count = len(sensor_data_df)
coverage_pct = (labeled_count / total_count) * 100

print(f"Labeled samples: {labeled_count:,}")
print(f"Total samples: {total_count:,}")
print(f"Coverage: {coverage_pct:.1f}%")

# Show sample of labeled data
print(f"\nSample of labeled data:")
labeled_sample = sensor_data_df[sensor_data_df['Activity'].notna()].head(10)
print(labeled_sample[['timestamp', 'Activity', 'accelerometerX', 'accelerometerY', 'accelerometerZ']])

Activity distribution:
Activity
None           6520
Walking        1060
Stairs Up       958
Stairs Down     347
Laying          281
Sitting         237
Standing        120
Name: count, dtype: int64

Labeling coverage:
Labeled samples: 3,003
Total samples: 9,523
Coverage: 31.5%

Sample of labeled data:
         timestamp     Activity  accelerometerX  accelerometerY  \
703  1770556169587  Stairs Down           -1.13          -10.29   
704  1770556169634  Stairs Down           -0.41          -17.71   
705  1770556169673  Stairs Down           -2.32          -11.78   
706  1770556169694  Stairs Down           -4.20          -11.40   
707  1770556169733  Stairs Down           -4.08          -10.04   
708  1770556169753  Stairs Down           -2.65           -8.98   
709  1770556169773  Stairs Down            0.01           -7.93   
710  1770556169793  Stairs Down            2.80           -7.56   
711  1770556169813  Stairs Down            4.48           -7.49   
712  1770556169833  Stairs 

In [48]:
# Quick validation before running the labeling
print("=== PRE-LABELING VALIDATION ===")

print(f"Labels to process:")
for i, row in labels_df.iterrows():
    duration_ms = row['EndTimestamp_Unix_Ms'] - row['StartTimestamp_Unix_Ms'] 
    duration_sec = duration_ms / 1000
    print(f"  {row['Label']}: {duration_sec:.1f}s ({row['StartTimestamp_Unix_Ms']} -> {row['EndTimestamp_Unix_Ms']})")

print(f"\nSensor data timestamp range:")
print(f"  First: {sensor_data_df['timestamp'].min()}")
print(f"  Last:  {sensor_data_df['timestamp'].max()}")

# Check for timestamp overlaps
print(f"\nChecking timestamp alignment...")
for i, row in labels_df.iterrows():
    sensor_count = ((sensor_data_df['timestamp'] >= row['StartTimestamp_Unix_Ms']) & 
                   (sensor_data_df['timestamp'] <= row['EndTimestamp_Unix_Ms'])).sum()
    print(f"  {row['Label']}: {sensor_count} sensor samples overlap")

print("\n" + "="*50)

=== PRE-LABELING VALIDATION ===
Labels to process:
  Stairs Down: 34.8s (1770556164324 -> 1770556199074)
  Walking: 33.7s (1770556208426 -> 1770556242166)
  Walking: 72.1s (1770557152264 -> 1770557224330)
  Stairs Up: 46.0s (1770557436738 -> 1770557482746)
  Standing: 35.5s (1770557597454 -> 1770557632976)
  Sitting: 50.7s (1770557635658 -> 1770557686372)
  Laying: 41.6s (1770557695440 -> 1770557737050)
  Stairs Down: 36.8s (1770561115962 -> 1770561152789)
  Stairs Up: 5.0s (1770561154802 -> 1770561159787)
  Stairs Up: 35.0s (1770561166589 -> 1770561201630)

Sensor data timestamp range:
  First: 1770556157833
  Last:  1770561221448

Checking timestamp alignment...
  Stairs Down: 445 sensor samples overlap
  Walking: 1259 sensor samples overlap
  Walking: 371 sensor samples overlap
  Stairs Up: 27 sensor samples overlap
  Standing: 275 sensor samples overlap
  Sitting: 340 sensor samples overlap
  Laying: 381 sensor samples overlap
  Stairs Down: 457 sensor samples overlap
  Stairs Up: 

In [53]:
clean_data = sensor_data_df.copy()
clean_data.dropna(subset=['Activity'], inplace=True)

clean_data.value_counts('Activity')

labels_mapping = {
    'Walking': 'WALKING',
    'Stairs Down': 'WALKING_DOWNSTAIRS',
    'Sitting': 'SITTING',
    'Standing': 'STANDING',
    'Lying': 'LAYING',
    'Stairs Up': 'WALKING_UPSTAIRS'
}

clean_data['Activity'] = clean_data['Activity'].map(labels_mapping)

clean_data.to_json(r"E:\src\neat-calculator\ml_monolith\data_preprocessing\labeled_sensor_data_08022026.json", orient='records', lines=True)